# 01 · Load data — structural extraction of the raw government spreadsheets

**Goal:** turn every presentation-formatted sheet into a tidy *long* table
**without any cleaning, filtering, imputation, or window restriction**. The only
transformations here are *structural* (find the real table, attach the right
category / date / region labels, melt to long). Missing-value tokens
(`---`, `///`, `-`, blank) are preserved verbatim in `raw_value`; numeric
coercion and missing-value classification happen in notebook `02`.

**Inputs** (read-only): `data/raw/Ehoba_02a_0811.xlsx`, `Ehoba_03a_0811.xlsx`
(rate/price), `Ehoba_04_0811.xlsx`, `Ehoba_VA_0811.xlsx`, `Ehoba_1_ano.xlsx`,
`sh_ipc_08_26.xls` (INDEC IPC, base dic-2016 = 100).

**Outputs** → `data/processed/01_loaded/`: one parquet per source table
(`room_occupancy`, `bed_occupancy`, `average_rate`, `travelers`, `capacity`, `cpi`).

See `DATA_AUDIT.md` §2 and §9 for the structure of each workbook.

In [1]:
import sys
from pathlib import Path

import openpyxl
import pandas as pd
import xlrd

SRC = Path.cwd() / "src"
sys.path.insert(0, str(SRC))
import config as C
from common import (AuditLog, canon_category, month_from_label, norm_text,
                    year_from_label)

LOG = AuditLog("01_load")
C.P_LOADED_DIR.mkdir(parents=True, exist_ok=True)
print("project root :", C.ROOT)
print("raw inputs   :", *[p.name for p in sorted(C.DATA_RAW.iterdir())])

project root : /Users/jaganathapandiyan/Desktop/Python/argentina-hotels
raw inputs   : Ehoba_02a_0811.xlsx Ehoba_03a_0811.xlsx Ehoba_04_0811.xlsx Ehoba_1_ano.xlsx Ehoba_VA_0811.xlsx sh_ipc_08_26.xls


## Monthly EHOBA sheets — generic block parser

Layout of `Ehoba_02a` / `03a` / `04` / `VA`: row 1 = title, a 2–3 row header
band, then repeating blocks of `[year row][12 month-name rows]`. Column A holds
the period label; data columns start at B. The category name for each data
column comes from the header row; the hotel / para-hotel *band* row
disambiguates the two columns both labelled "Total".

In [2]:
def _band_forward_fill(ws, row, c_lo, c_hi):
    """{col -> band label}, filling merged-anchor values across columns."""
    out, cur = {}, ""
    for c in range(c_lo, c_hi + 1):
        v = norm_text(ws.cell(row=row, column=c).value)
        if v:
            cur = v.lower()
        out[c] = cur
    return out


def load_monthly_ehoba(path, sheet, *, header_row, band_row, first_data_row,
                       c_lo=2, c_hi=None, total_is_parahotel_band=True):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb[sheet]
    c_hi = c_hi or ws.max_column
    bands = (_band_forward_fill(ws, band_row, c_lo, c_hi)
             if band_row else {c: "" for c in range(c_lo, c_hi + 1)})

    col_cat = {}
    for c in range(c_lo, c_hi + 1):
        raw = norm_text(ws.cell(row=header_row, column=c).value)
        if not raw:
            continue
        cat = canon_category(raw)
        if cat == "total_hoteleros" and total_is_parahotel_band and \
                "parahotel" in bands.get(c, ""):
            cat = C.CAT_TOTAL_PARAHOTEL
        if cat:
            col_cat[c] = cat
        else:
            LOG.log("unmapped_header", f"{sheet} col{c}", detail=repr(raw))

    records, cur_year = [], None
    n_year = n_month = 0
    for r in range(first_data_row, ws.max_row + 1):
        a = ws.cell(row=r, column=1).value
        y = year_from_label(a)
        if y is not None and all(ws.cell(row=r, column=c).value in (None, "")
                                 for c in list(col_cat)[:3]):
            cur_year, n_year = y, n_year + 1
            continue
        m = month_from_label(a)
        if m is None or cur_year is None:
            continue
        n_month += 1
        prov = "*" in str(a)
        for c, cat in col_cat.items():
            raw = ws.cell(row=r, column=c).value
            records.append({
                "date": pd.Timestamp(cur_year, m, 1),
                "hotel_category": cat,
                "raw_value": "" if raw is None else str(raw),
                "cell_value": float(raw) if isinstance(raw, (int, float)) else float("nan"),
                "cell_is_numeric": isinstance(raw, (int, float)),
                "provisional": prov,
                "src_sheet": sheet, "src_row": r, "src_col": c,
            })
    wb.close()
    df = pd.DataFrame.from_records(records)
    LOG.log("parsed", f"{path.name}:{sheet}", len(df),
            f"{n_year} year-rows, {n_month} month-rows, "
            f"{df['hotel_category'].nunique()} categories, "
            f"{df['date'].min():%Y-%m}..{df['date'].max():%Y-%m}")
    return df

In [3]:
room = load_monthly_ehoba(C.F_ROOM_OCC, "Ehoba_02a_0811",
                          header_row=4, band_row=3, first_data_row=5)
bed = load_monthly_ehoba(C.F_BED_OCC, "Ehoba_04_0811",
                         header_row=4, band_row=3, first_data_row=5)
rate = load_monthly_ehoba(C.F_RATE, "Ehoba_03a_0811",
                          header_row=4, band_row=3, first_data_row=5)
trav = load_monthly_ehoba(C.F_TRAVELERS, "Ehoba_VA_0811",
                          header_row=3, band_row=None, first_data_row=4,
                          total_is_parahotel_band=False)
rate.head()

  [01_load] parsed                 Ehoba_02a_0811.xlsx:Ehoba_02a_0811     2210  19 year-rows, 221 month-rows, 10 categories, 2008-01..2026-05


  [01_load] parsed                 Ehoba_04_0811.xlsx:Ehoba_04_0811     2210  19 year-rows, 221 month-rows, 10 categories, 2008-01..2026-05


  [01_load] parsed                 Ehoba_03a_0811.xlsx:Ehoba_03a_0811     1768  19 year-rows, 221 month-rows, 8 categories, 2008-01..2026-05


  [01_load] parsed                 Ehoba_VA_0811.xlsx:Ehoba_VA_0811      966  14 year-rows, 161 month-rows, 6 categories, 2013-01..2026-05


,date,hotel_category,raw_value,cell_value,cell_is_numeric,provisional,src_sheet,src_row,src_col
0,2008-01-01,stars_1_2,97.17045454545455,97.170455,True,False,Ehoba_03a_0811,6,2
1,2008-01-01,stars_3,162.31034482758622,162.310345,True,False,Ehoba_03a_0811,6,3
2,2008-01-01,stars_4,230.31147540983608,230.311475,True,False,Ehoba_03a_0811,6,4
3,2008-01-01,stars_5,581.9130434782609,581.913043,True,False,Ehoba_03a_0811,6,5
4,2008-01-01,apart,202,202.000000,True,False,Ehoba_03a_0811,6,6


## Capacity workbook (`Ehoba_1_ano.xlsx`)

One sheet per year; each sheet has 1–4 **quarterly snapshot blocks**
(March / June / July / Sept / Dec depending on year). Each block: a month
marker row, a category header row, then three metric rows — establishments,
`Habitaciones o unidades disponibles` (= rooms × days-open = **room-nights**),
`Plazas disponibles` (= **bed-nights**). The 2008 sheet uses the older EOH
taxonomy → tagged `source_taxonomy = "2008_eoh"`.

In [4]:
CAP_METRIC = {
    "establecimientos": "establishments",
    "hoteles": "establishments",
    "habitaciones o unidades disponibles": "available_room_nights",
    "plazas disponibles": "available_bed_nights",
}
CAP_MONTHS = dict(C.MONTHS_ES_UPPER)


def _cap_metric(label):
    s = norm_text(label).lower().rstrip(".")
    s = "".join(ch for ch in s if not ch.isdigit()).strip()
    for k, v in CAP_METRIC.items():
        if s.startswith(k):
            return v
    return None


def load_capacity(path):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    records = []
    for sheet in wb.sheetnames:
        if not sheet.strip().isdigit():
            continue
        year = int(sheet.strip())
        ws = wb[sheet]
        taxonomy = "2008_eoh" if year == 2008 else "ehoba"
        cur_month, col_cat, band = None, {}, {}
        for r in range(1, ws.max_row + 1):
            rowvals = [ws.cell(row=r, column=c).value for c in range(1, ws.max_column + 1)]
            for v in rowvals:
                mt = norm_text(v).upper().rstrip(" .")
                if mt in CAP_MONTHS:
                    cur_month = CAP_MONTHS[mt]
                    break
            joined = " ".join(norm_text(v).lower() for v in rowvals)
            if "hoteleros" in joined and "estrellas" not in joined:
                band = _band_forward_fill(ws, r, 1, ws.max_column)
            cats = {c: canon_category(norm_text(ws.cell(row=r, column=c).value))
                    for c in range(1, ws.max_column + 1)}
            mappable = {c: v for c, v in cats.items() if v}
            if len(mappable) >= 3:
                col_cat = {}
                for c, v in mappable.items():
                    if v == "total_hoteleros" and "parahotel" in band.get(c, ""):
                        v = C.CAT_TOTAL_PARAHOTEL
                    col_cat[c] = v
                continue
            metric = _cap_metric(ws.cell(row=r, column=1).value)
            if metric and col_cat and cur_month:
                for c, cat in col_cat.items():
                    raw = ws.cell(row=r, column=c).value
                    records.append({
                        "date": pd.Timestamp(year, cur_month, 1),
                        "snapshot_month": cur_month, "hotel_category": cat,
                        "metric": metric,
                        "raw_value": "" if raw is None else str(raw),
                        "cell_value": float(raw) if isinstance(raw, (int, float)) else float("nan"),
                        "cell_is_numeric": isinstance(raw, (int, float)),
                        "source_taxonomy": taxonomy,
                        "src_sheet": sheet, "src_row": r, "src_col": c,
                    })
    wb.close()
    df = pd.DataFrame.from_records(records).drop_duplicates(
        subset=["date", "hotel_category", "metric"])
    LOG.log("parsed", path.name, len(df),
            f"{df['date'].dt.year.nunique()} years, {sorted(df['metric'].unique())}, "
            f"{df['date'].min():%Y-%m}..{df['date'].max():%Y-%m}")
    return df


cap = load_capacity(C.F_CAPACITY)
cap.groupby(cap.date.dt.year).date.apply(lambda s: sorted(s.dt.month.unique()))

  [01_load] parsed                 Ehoba_1_ano.xlsx                     1830  19 years, ['available_bed_nights', 'available_room_nights', 'establishments'], 2008-03..2026-03


date
2008          [3, 12]
2009          [3, 12]
2010          [3, 12]
2011       [3, 7, 12]
2012       [3, 7, 12]
2013       [3, 7, 12]
2014       [3, 7, 12]
2015       [3, 7, 12]
2016    [3, 6, 9, 12]
2017    [3, 6, 9, 12]
2018    [3, 6, 9, 12]
2019    [3, 6, 9, 12]
2020    [3, 6, 9, 12]
2021    [3, 6, 9, 12]
2022    [3, 6, 9, 12]
2023    [3, 6, 9, 12]
2024    [3, 6, 9, 12]
2025    [3, 6, 9, 12]
2026              [3]
Name: date, dtype: object

## CPI workbook (`sh_ipc_08_26.xls`) — INDEC IPC, base dic-2016 = 100

`.xls` (BIFF) → `xlrd`. Periods run **across columns** as Excel date serials;
region blocks (`Total nacional`, `Región GBA`, Pampeana, …) are stacked down
column A, each starting with a header row that carries the date vector. We keep
the index-level sheet and the MoM sheet (for a cross-check), plus the GBA
bridge sheet that extends GBA back to 2016-04.

In [5]:
CPI_REGIONS = {
    "total nacional": "total_nacional",
    "región gba": "gba", "region gba": "gba",
    "región pampeana": "pampeana", "region pampeana": "pampeana",
    "región noroeste": "noa", "region noroeste": "noa", "noa": "noa",
    "región noreste": "nea", "region noreste": "nea", "nea": "nea",
    "región cuyo": "cuyo", "region cuyo": "cuyo",
    "región patagonia": "patagonia", "region patagonia": "patagonia",
}


def _xlrd_dates(sh, r, datemode):
    out = {}
    for c in range(1, sh.ncols):
        v = sh.cell_value(r, c)
        if isinstance(v, (int, float)) and v > 30000:
            out[c] = pd.Timestamp(xlrd.xldate.xldate_as_datetime(v, datemode))
    return out


def _parse_cpi_stacked(sh, datemode, sheet_name):
    rec, region, datevec = [], None, {}
    for r in range(sh.nrows):
        a = norm_text(sh.cell_value(r, 0))
        if a.lower() in CPI_REGIONS:
            region = CPI_REGIONS[a.lower()]
            datevec = _xlrd_dates(sh, r, datemode)
            continue
        if not a or region is None or not datevec:
            continue
        vals = {c: sh.cell_value(r, c) for c in datevec
                if isinstance(sh.cell_value(r, c), (int, float))
                and sh.cell_value(r, c) != ""}
        for c, v in vals.items():
            rec.append({"src_sheet": sheet_name, "region": region,
                        "series": a, "date": datevec[c], "value": float(v)})
    return rec


def load_cpi(path):
    wb = xlrd.open_workbook(path)
    dm = wb.datemode
    rec = _parse_cpi_stacked(wb.sheet_by_name(C.CPI_SHEET_INDEX), dm, "index_national")
    rec += _parse_cpi_stacked(wb.sheet_by_name(C.CPI_SHEET_MOM), dm, "mom_national")
    sh = wb.sheet_by_name(C.CPI_SHEET_GBA_BRIDGE)
    for r in range(sh.nrows):
        if norm_text(sh.cell_value(r, 0)).lower() in ("descripción", "descripcion", "apertura"):
            datevec = _xlrd_dates(sh, r, dm)
            for rr in range(r + 1, sh.nrows):
                lbl = norm_text(sh.cell_value(rr, 0))
                if not lbl:
                    continue
                if lbl.lower() in ("apertura", "descripción", "descripcion"):
                    break
                for c in datevec:
                    v = sh.cell_value(rr, c)
                    if isinstance(v, (int, float)):
                        rec.append({"src_sheet": "gba_bridge", "region": "gba",
                                    "series": lbl, "date": datevec[c], "value": float(v)})
            break
    df = pd.DataFrame.from_records(rec)
    LOG.log("parsed", path.name, len(df),
            f"sheets={sorted(df['src_sheet'].unique())}, "
            f"regions={sorted(df['region'].unique())}, "
            f"{df['date'].min():%Y-%m}..{df['date'].max():%Y-%m}")
    return df


cpi = load_cpi(C.F_CPI)
cpi[(cpi.src_sheet == "index_national") & (cpi.region == "gba") &
    (cpi.series.str.lower() == "nivel general")].tail(3)

  [01_load] parsed                 sh_ipc_08_26.xls                    29448  sheets=['gba_bridge', 'index_national', 'mom_national'], regions=['cuyo', 'gba', 'nea', 'noa', 'pampeana', 'patagonia', 'total_nacional'], 2016-04..2026-07


,src_sheet,region,series,date,value
2201,index_national,gba,Nivel general,2026-05-01,11594.5499
2202,index_national,gba,Nivel general,2026-06-01,11810.9464
2203,index_national,gba,Nivel general,2026-07-01,12078.7372


## Write outputs & flush the audit log

In [6]:
room.to_parquet(C.P_LOADED_DIR / "room_occupancy.parquet")
bed.to_parquet(C.P_LOADED_DIR / "bed_occupancy.parquet")
rate.to_parquet(C.P_LOADED_DIR / "average_rate.parquet")
trav.to_parquet(C.P_LOADED_DIR / "travelers.parquet")
cap.to_parquet(C.P_LOADED_DIR / "capacity.parquet")
cpi.to_parquet(C.P_LOADED_DIR / "cpi.parquet")
LOG.log("write", str(C.P_LOADED_DIR.relative_to(C.ROOT)), detail="6 parquet files")
LOG.flush()
print("01_load_data complete.")

  [01_load] write                  data/processed/01_loaded                   6 parquet files
01_load_data complete.
